# ICS 604: APPLIED DATA SCIENCE

## Maximum Likelihood Parameter Estimation
---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Maximum Likelihood Parameter Estimation

Recall the example: To estimate the rate of moving traffic citations near the UH Mānoa, we randomly count the number of observed citations issued within a 5-mile radius of campus over 90 days distributed across all 12 months of the year. Spreading the sampling period throughout the calendar year helps reduce seasonal bias. For example, citation counts may decrease during the summer when fewer students are on campus, or increase during the holiday season when traffic patterns change. By evenly distributing observation days, we aim to obtain a representative sample of citation activity and produce a more reliable estimate of the underlying rate.

Estimating the distribution parameter using maximum likelihood means choosing the parameter value that makes the observed data most probable under an assumed statistical model. For instance, if we assume the number of daily moving traffic citations follows a Poisson distribution with rate parameter $\lambda$, the maximum likelihood estimate (MLE) of $\lambda$ is the value that maximizes the likelihood function given our 90 days of observed counts. Intuitively, this results in selecting the parameter that best “fits” the data — under the Poisson assumption, the MLE for $\lambda$ is simply the sample mean number of citations per day.

In [ ]:
citations_data = pd.read_csv("data/citations_counts.tsv", index_col="Day")
citations_data.head()

In [ ]:
citations_data.shape

In [ ]:
plt.figure(figsize=(6, 3))
plt.hist(citations_data["Counts"], bins=8, density=True, 
         edgecolor='black', linewidth=1.2, alpha=0.5);

# x = np.arange(30)

### Fitting the Data

Because the number of moving traffic citations per day is count data (non-negative integers), a Poisson distribution is a natural model to consider. From the observed data, the most frequent daily count appears to be approximately 17 citations. This suggests that the underlying mean rate of the process is likely near 17–18.

A Poisson distribution with mean $\lambda = 9$ would place most of its probability mass around 9 citations per day, with rapidly decreasing probability for much larger counts like 17. Therefore, it would be unlikely to generate a dataset whose most common value is near 17. In contrast, a Poisson distribution with $\lambda = 17$ centers its mass around 17 and assigns relatively high probability to values near 17, making it far more consistent with the observed data.
  
<center><img src="https://www.dropbox.com/scl/fi/cbnto3fj3v4mldl1z2ak0/counts_lambdas.png?rlkey=e1sox73r3k0p0widpfo558th9&st=4u9vpeg5&dl=1" alt="drawing" width="450px"></center>

#### Question:
Why do the two distributions have different heights?

In [ ]:
from scipy.stats import poisson

np.random.seed(4)

plt.figure(figsize=(6, 3))
plt.hist(citations_data["Counts"], bins=8, density=True, edgecolor='black', linewidth=1.2)
x = np.arange(30)

p_y = poisson.pmf(x, 9)
plt.plot(x, p_y, color='r', linewidth=4, label="$\\lambda=9$")
p_y = poisson.pmf(x, 17)
plt.plot(x, p_y, color='k', linewidth=4, label="$\\lambda=17$")
plt.legend();

In [ ]:
plt.figure(figsize=(6, 3))
plt.hist(citations_data["Counts"], bins=8, density=True, 
         edgecolor='black', linewidth=1.2, alpha=0.5)

for i in range(12, 21):
    p_y = poisson.pmf(x, i)
    plt.plot(x, p_y, color='r', linewidth=0.5, alpha=.9)

p_y = poisson.pmf(x, 17)
plt.plot(x, p_y, color='b', linewidth=2)
plt.show()

#### Question:
The graph above suggests that some plots fit the data better than others. How could we formally (or informally) decide which one fits the best?

### Computing the Probability of Observing These Data 

Once we assume that daily citation counts follow a Poisson distribution with parameter $\lambda$, we can compute the probability of each observed data point using the probability mass function (pmf). For a single observation $x$, this is:

```python
    poisson.pmf(x, lambda)
```
If we observe counts over many days (say $x_1, x_2,...,x_n$), and we assume the days are **independent**, then the probability of observing the entire dataset under a given $\lambda$ is the product of the individual probabilities:

```python
    np.prod(poisson.pmf(x, lambda))
```
 
We can compute the probability of observing the data for each possible value of the parameter $\lambda$. In practice, however, we restrict our attention to a range of reasonable values rather than considering all possible numbers. This range is typically guided by the observed data — for example, values near the sample mean are more plausible than extremely small or extremely large ones. By evaluating the likelihood over this reasonable set of candidate values, we can determine which $\lambda$ makes the observed dataset most probable.

#### Example: Compound probability of Independent Events

The idea is similar to flipping a fair coin. If
$$
p(H) = p(T) = 1/2,
$$ 
and we flip the coin twice, the sample space is

$$
\{HH, HT, TH, TT\}.
$$

Because the flips are independent, probabilities of the different outcomes are:

$$
\begin{align*}
p(HH) &= p(H) \times p(H) = 1/4, \\
p(HT) &= p(H) \times p(T) = 1/4, \\
p(TH) &= p(T) \times p(H) = 1/4, \\
p(TT) &= p(T) \times p(T) = 1/4.
\end{align*}
$$

All probabilities sum to 1, as expected. 

This multiplication rule works because the events are independent — the outcome of the first flip does not influence the second.

Similarly, in our traffic citation example, we assume that the number of citations on one day does not affect the number on another day. Under this independence assumption, the joint probability of observing all 90 days of data is the product of the daily Poisson probabilities.

### Computing the Likelihood of Observing Data

When the events are independent, the probability of observing the entire dataset is simply the product of the probabilities of the individual observations. This combined probability is called the **likelihood** $L$ of the data given the parameter $\lambda$. For example, if we have 90 days of observed traffic citation counts $[x_1, x_2, ..., x_{90}]$, the likelihood is computed as:

$$ 
\begin{align}
L([x_1,x_2,...,x_{90}] | \lambda) 
  &= \mbox{pmf}(x_1, \lambda) \times \mbox{pmf}(x_2, \lambda) \times \cdots \times \mbox{pmf}(x_{90}, \lambda) \\
  &= \prod_{i=1}^{90} \mbox{pmf}(x_i, \lambda) 
\end{align}
$$

This formula expresses the probability of the complete dataset under the assumption that each day’s citation count is independent and follows the same Poisson distribution with mean $\lambda$.

In [ ]:
data_point = 12
_lambda = 16
poisson.pmf(data_point, _lambda)

In [ ]:
data_point = 16
_lambda = 16
poisson.pmf(data_point, _lambda)

In [ ]:
poisson.pmf(citations_data["Counts"], _lambda)

In [ ]:
# Likelihood of the citation data

print(np.prod(poisson.pmf(citations_data["Counts"], _lambda)))

#### Question:
How can the likelihood of citation data be so low?

### Understanding the Likelihood

The likelihood of observing a specific dataset can be extremely small, and this is normal. Consider a simple example of just three days with citation counts of (16, 15, 21). The total sample space includes all possible combinations of counts for those three days, such as (0, 1, 8), (1, 10, 10), (21, 11, 28), and many others. If the daily count can range from 0 to 50, there are $51^3 = 132,651$ possible outcomes. The observed sequence (16, 15, 21) represents just one of these many possibilities, so its probability is necessarily very small.

For 90 days of observations, the sample space becomes astronomically large — on the order of $4.8 \times 10^{153}$ possible sequences. With so many potential outcomes, it is entirely expected that the computed likelihood for the observed dataset is an extremely small number. This small value does not indicate that the model is incorrect; it simply reflects the vast number of possible outcomes in the sample space.

In [ ]:
print(51**3)
print(f"{pow(51, 90):.10e}")

In [ ]:
_lambda = 16
print(np.prod(poisson.pmf((16, 15, 21), _lambda)))

In [ ]:
print(np.prod(poisson.pmf((9, 11, 11), _lambda)))

In [ ]:
total_prob = 0
for i in range(0, 51):
    for j in range(0, 51):
        for k in range(0, 51):
            total_prob += np.prod(poisson.pmf((i, j, k), _lambda))
            
print(total_prob)

In [ ]:
for i in range(12, 21):
    likelihood_data_i = np.prod(poisson.pmf(citations_data["Counts"], i))
    print(f"The likelihood of the data given λ = {i} is {likelihood_data_i:.5e}")

In [ ]:
plt.figure(figsize=(6, 3))

x = range(12, 21)
l_x = [np.prod(poisson.pmf(citations_data["Counts"], i)) for i in x]

plt.scatter(x, l_x)
plt.ylim(-1.1e-113, 0.2e-112)
plt.show()

When computing the likelihood of a large dataset, the result can become extremely small. This is because we are multiplying many probabilities together, each of which is less than one. In practice, working with such small numbers can cause floating-point underflow, where the computed value is smaller than what the computer can accurately represent in memory. To avoid this issue, we typically work with the **log-likelihood** instead.

### Log-likelihood

The log-likelihood transforms the product of probabilities into a sum, taking advantage of the property:

$$
\mbox{log}(x \cdot y) = \mbox{log}(x) + \mbox{log}(y)
$$

For our 90-day dataset, the log-likelihood is:

\begin{align}
\mbox{log}(L([x_1,x_2,..., x_{90}] | \lambda))
&= \mbox{log}(\mbox{pmf}(x_1, \lambda) \times  \mbox{pmf}(x_2, \lambda) \times \cdots \times \mbox{pmf}(x_{90}, \lambda)) \\
&= \mbox{log}(\mbox{pmf}(x_1, \lambda)) + \mbox{log}(\mbox{pmf}(x_2, \lambda)) + \cdots + \mbox{log}(\mbox{pmf}(x_{90}, \lambda)) \\
&= \sum_{i=1}^{90} \mbox{log}(\mbox{pmf}(x_i, \lambda)) 
\end{align}

Using the log-likelihood prevents underflow and is numerically more stable. It is important to note that log-likelihood values are not probabilities; they do not sum to 1. Instead, they provide a measure that preserves the shape of the likelihood function and allows us to identify the value of $\lambda$ that maximizes the probability of observing the data.

In [ ]:
x = range(12, 21)
l_x = [np.sum(np.log(poisson.pmf(citations_data["Counts"], i))) for i in x]

plt.figure(figsize=(6, 3))
plt.scatter(x, l_x)
plt.show()

### Maximum Likelihood for the Poisson Distribution

In maximum likelihood estimation (MLE), we hypothesize that the observed data were generated from a Poisson distribution with some unknown parameter $\lambda$. Among all possible values of $\lambda$, there is one that makes the observed data most probable—this is the value that maximizes the likelihood.

For the Poisson distribution, it can be shown mathematically that the likelihood is maximized when $\lambda$ equals the sample mean of the observed counts. For example, if we calculate the mean of our daily citation counts:

```python 
>>> np.mean(citations_data["Counts"]) 
16.41
```

the maximum likelihood estimate of $\lambda$ is 16.41.

While we assumed a Poisson distribution for this dataset, the same principle can be applied to other candidate distributions. By computing and comparing likelihoods for different models, we can select the distribution that best explains the data. This makes maximum likelihood a powerful tool not only for estimating parameters but also for model selection and decision-making.

## Example: A/B Testing

In an A/B test, we compare two versions of a webpage to see which performs better at a specific task $X$. For example, we might want to determine whether Version A (new) or Version B (old) is more effective at getting visitors to sign up for a newsletter.

<center><img src="https://www.dropbox.com/scl/fi/rnvqswb1yjkkvvlvsouk0/ab.jpg?rlkey=twogus8zpr53zpq9u59fgto5r&st=6qq3jngj&dl=1" alt="drawing" style="width:600px"></center>

### Maximum Likelihood for a Binomial Distribution

Focusing on Version A, suppose we test 8 individuals (an unrealistically small sample, used here for illustration) and observe that 5 of them sign up. From this outcome, we can make several inferences about the underlying data and its generating process:

1. The data are Binomially distributed, with $n=8$ trials and a success probability $p$ (the probability that a user signs up) somewhere between 0 and 1.

2. Intuitively, the observed outcome tells us that $p$ cannot be extremely small or extremely large. For example, seeing 5 sign-ups out of 8 is far more consistent with a probability like $p=0.8$ than $p=0.03$. The likelihood of the observed data is higher for intermediate-to-high values of $p$.

We can estimate the most likely value of $p$ using the same maximum likelihood approach as with the traffic citation data. For a Binomial distribution, the MLE of $p$ is simply the proportion of successes:

$$
\hat{p} = \frac{\text{number of successes}}{n} = \frac{5}{8} = 0.625
$$

Thus, based on this small sample, the best estimate of the probability that a visitor signs up using Version A is 0.625. This illustrates how MLE can be applied to different types of data and distributions beyond the Poisson case.

In [ ]:
from scipy.stats import binom

n = 8 
p = 0.5
binom.pmf(3, n ,p)

In [ ]:
5/8

In [ ]:
binom.pmf(4, n ,p)

In [ ]:
x = np.arange(0.025, 1, 0.025)
llh_x_vals = []

for p in x:
    # data size = 1. No need to sum in this example.
    llh_x_i = np.sum(np.log(binom.pmf(5, n, p))) 
    llh_x_vals.append(llh_x_i)

mle_p = x[np.argmax(llh_x_vals)]
print(f"MLE of p = {mle_p}")

plt.figure(figsize=(6, 4))
plt.scatter(x, llh_x_vals)
plt.scatter(x[np.argmax(llh_x_vals)], np.max(llh_x_vals))
plt.axvline(mle_p, color='r')
plt.show()

In [ ]:
x = np.arange(0.025, 1, 0.025)
llh_x_vals = []

for p in x:
    llh_x_i = np.log(binom.pmf(5, n, p)) # data size = 1. No need to sum in this example.
    llh_x_vals.append(llh_x_i)

mle_p = x[np.argmax(llh_x_vals)]
print(f"MLE of p = {mle_p}")

plt.figure(figsize=(6, 4))
plt.scatter(x, llh_x_vals)
plt.scatter(x[np.argmax(llh_x_vals)], np.max(llh_x_vals))
plt.axvline(mle_p, color='r')
plt.show()

##  Maximum Likelihood Estimation

Maximum likelihood estimation provides a systematic way to estimate parameters of common probability distributions. For many standard distributions, the MLE can be computed analytically. For example, for a Gaussian (normal) distribution, the maximum likelihood estimator of the mean is simply the sample mean, and the estimator of the standard deviation is the sample standard deviation:
     
$$
\hat{\mu} = \frac{1}{n} \sum_{i=1}^{n}x_i
$$

$$
\hat{\sigma} = \sqrt{\frac{1}{n} \sum_{i=1}^{n}{(x_i - \bar{x})^2}}
$$

MLEs are **point estimates** that maximize the likelihood of observing the given data. Importantly, they do not provide a probability that the estimated value is the “true” parameter of the population, nor do they inherently provide confidence intervals, unlike approaches such as the bootstrap.

This raises a natural question: **“How different could this estimate have been if we had observed a different sample?”** Understanding the variability of MLEs across hypothetical samples is key to assessing the reliability of the estimate, which often requires additional tools such as confidence intervals, standard errors, or resampling methods.